In [1]:
import os
import pandas as pd
import sys
from pathlib import Path

project_root = os.path.abspath("..")   # lên 1 cấp: MIND-research

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from src.core.Context import VectorContext
from src.urv.URV import URV
from src.recommendation.RecommendationEngine import RecommendationEngine
from src.matrix.Matrix import Metrix
import numpy as np

In [3]:
import pickle

context = VectorContext(
    os.path.join(project_root, "data", "primary", "test_set")
)

vectors_dir = os.path.join(project_root, "vectors", "primary")
represent_vectors_path = os.path.join(vectors_dir, "represent_vectors.pkl")
with open(represent_vectors_path, "rb") as f:
    represented_vector = pickle.load(f)

urv = URV()
recommender = RecommendationEngine(alpha=0.75)


In [4]:
from pathlib import Path

results_dir = Path(project_root) / "results"
results_dir.mkdir(exist_ok=True)

results_path = results_dir / "ranked_results.jsonl"

# Tạo file mới
results_path.write_text("", encoding="utf-8")

print(f"Saving to: {results_path}")

Saving to: d:\CDNC\MIND-research\results\ranked_results.jsonl


In [8]:
import gc
import json
import os

# ==========================================
# Configuration
# ==========================================

processed = 0
missing_vectors = set()

available_news_ids = set(represented_vector)

# XÓA file test cũ
if results_path.exists():
    results_path.unlink()

print("Starting FULL test run...")


# ==========================================
# Process first 5000 behaviors
# ==========================================
for behaviours in context.createImpressedNoLabel(chunksize=5000):

    batch_results = []

    for behavior_id, behavior in behaviours.items():
        # ----------------------------------
        # 1. History
        # ----------------------------------

        history_ids = behavior["history"].split()

        valid_history_ids = [
            news_id
            for news_id in history_ids
            if news_id in available_news_ids
        ]

        missing_history_ids = (
            set(history_ids) - set(valid_history_ids)
        )

        # ----------------------------------
        # 2. Candidates
        # ----------------------------------

        candidate_news = []
        missing_candidates = []

        for impression in behavior["impressions"]:

            news_id = impression["news_id"]

            if news_id not in available_news_ids:

                missing_candidates.append({
                    "news_id": news_id,
                })

                continue

            candidate_news.append({
                "news_id": news_id,
                "vector": represented_vector[news_id],
            })

        # ----------------------------------
        # 3. Track missing vectors
        # ----------------------------------

        missing_ids = (
            missing_history_ids |
            {
                item["news_id"]
                for item in missing_candidates
            }
        )

        if missing_ids:
            missing_vectors.update(missing_ids)

        # ----------------------------------
        # 4. Ranking
        # ----------------------------------

        ranked_candidates = []

        if valid_history_ids and candidate_news:

            valid_history = " ".join(valid_history_ids)

            user_vector = urv.getURV(
                valid_history,
                represented_vector,
            )

            ranked_candidates = recommender.recommended_no_label(
                user_vector=user_vector,
                candidate_news=candidate_news,
            )

        else:
            # Cold-start: no valid history
            # Preserve original candidate order
            ranked_candidates = [
                {
                    "news_id": news["news_id"],
                    "score": 0.0
                }
                for news in candidate_news
            ]

        # ----------------------------------
        # 5. Append candidates without vectors
        # ----------------------------------
        ranked_candidates.extend(
            {
                "news_id": item["news_id"],
                "score": 0.0
            }
            for item in missing_candidates
        )

        # ----------------------------------
        # 6. Save result
        # ----------------------------------

        batch_results.append({
            "behavior_id": behavior_id,
            "ranking": ranked_candidates,
        })

        processed += 1

    # ======================================
    # Save batch
    # ======================================

    if batch_results:

        with open(
            results_path,
            "a",
            encoding="utf-8"
        ) as f:

            for result in batch_results:

                f.write(
                    json.dumps(
                        result,
                        ensure_ascii=False
                    ) + "\n"
                )

    # ======================================
    # Preview
    # ======================================

    if batch_results:

        sample = batch_results[-1]

        preview = " | ".join(
            f"{item['news_id']}:"
            f"{item.get('score', 'NA')}"
            for item in sample["ranking"][:5]
        )

        print(
            f"[{processed:,}] "
            f"{sample['behavior_id']} → "
            f"{preview} ..."
        )

    # ======================================
    # Release RAM
    # ======================================

    del behaviours
    del batch_results

    gc.collect()


# ==========================================
# Final
# ==========================================

print("\n===== FULL TEST COMPLETE =====")
print(f"Processed: {processed:,}")
print(f"Missing vector IDs: {len(missing_vectors):,}")
print(f"Output: {results_path}")

Starting FULL test run...
[5,000] 5000 → N9678:0.32928703051185143 | N79482:0.2968937774247725 | N101247:0.2891893725525544 | N61652:0.28625838242124285 | N54448:0.25051216944582155 ...
[10,000] 10000 → N125441:0.4671846348445503 | N74979:0.3087323983058108 | N24602:0.3035238812175354 | N85514:0.23281359721540748 | N85077:0.22976585290332843 ...
[15,000] 15000 → N63136:0.21982166024931069 | N43496:0.20130156554040168 | N65220:0.13092413681895979 | N91314:0.1056818079846767 ...
[20,000] 20000 → N28172:0.19015782969636774 | N36143:0.18027667780235337 ...
[25,000] 25000 → N22562:0.5120986727346301 | N88716:0.4945813345651901 | N8043:0.4097161323277785 | N34085:0.4025905731100967 | N90685:0.402293638118688 ...
[30,000] 30000 → N5591:0.46861516437248024 | N84746:0.3517332638248915 | N57092:0.3293731030301013 | N37079:0.2976743114815325 | N39193:0.29706992539220733 ...
[35,000] 35000 → N17568:0.27204327314668464 | N124481:0.23276424039063223 | N80760:0.21925537785250254 | N9923:0.21114625019